# Experiment: Inspecting Nested Data

In [2]:

from datasets import load_dataset
import pandas as pd

# Load only the first 10 examples to save time
dataset = load_dataset("allenai/qasper", split="train[:10]", trust_remote_code=True)
sample = dataset[0]

# Exploring Scientific papers multiple sections
print(f"Paper Title: {sample['title']}")
print(f"Number of Sections: {len(sample['full_text']['section_name'])}")


/home/nguegnang/anaconda3/envs/qasper-rag/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Paper Title: Minimally Supervised Learning of Affective Events Using Discourse Relations
Number of Sections: 21


In [3]:
# Let's verify if paragraphs align with sections
sections = sample['full_text']['section_name']
paragraphs = sample['full_text']['paragraphs']

df = pd.DataFrame({
    "Section": sections,
    "Paragraph_Count": [len(p) for p in paragraphs],
    "First_Para_Preview": [p[0][:50] + "..." if p else "N/A" for p in paragraphs]
})

print(df.head(7))

                                             Section  Paragraph_Count  \
0                                       Introduction                4   
1                                       Related Work                5   
2                                    Proposed Method                1   
3              Proposed Method ::: Polarity Function                4   
4  Proposed Method ::: Discourse Relation-Based E...                3   
5  Proposed Method ::: Discourse Relation-Based E...                2   
6  Proposed Method ::: Discourse Relation-Based E...                2   

                                  First_Para_Preview  
0  Affective events BIBREF0 are events that typic...  
1  Learning affective events is closely related t...  
2                                                ...  
3                                                ...  
4  Our method requires a very small seed lexicon ...  
5  The seed lexicon matches (1) the latter event ...  
6  The seed lexicon matches ne

In [4]:
# Let's see the first actual text
if len(paragraphs[0]) > 0:
    print(f"Text Content: {paragraphs[0][0][:100]}...")

Text Content: Affective events BIBREF0 are events that typically affect people in positive or negative ways. For e...


In [15]:
print(type(dataset))   # <class 'datasets.arrow_dataset.Dataset'>


<class 'datasets.arrow_dataset.Dataset'>


In [16]:
print(dataset)          # Shows schema, num rows, and features


Dataset({
    features: ['id', 'title', 'abstract', 'full_text', 'qas', 'figures_and_tables'],
    num_rows: 10
})


In [17]:
print(dataset.features) # Shows column names and their types

{'id': Value(dtype='string', id=None), 'title': Value(dtype='string', id=None), 'abstract': Value(dtype='string', id=None), 'full_text': Sequence(feature={'section_name': Value(dtype='string', id=None), 'paragraphs': [Value(dtype='string', id=None)]}, length=-1, id=None), 'qas': Sequence(feature={'question': Value(dtype='string', id=None), 'question_id': Value(dtype='string', id=None), 'nlp_background': Value(dtype='string', id=None), 'topic_background': Value(dtype='string', id=None), 'paper_read': Value(dtype='string', id=None), 'search_query': Value(dtype='string', id=None), 'question_writer': Value(dtype='string', id=None), 'answers': Sequence(feature={'answer': {'unanswerable': Value(dtype='bool', id=None), 'extractive_spans': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), 'yes_no': Value(dtype='bool', id=None), 'free_form_answer': Value(dtype='string', id=None), 'evidence': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), 'highlighted_e

In [22]:
# Inspect Q&A structure across multiple samples
for i in range(min(3, len(dataset))):
    paper = dataset[i]
    qas = paper['qas']
    print(f"\n--- Paper {i}: {paper['title'][:80]} ---")
    print(f"  Sections: {len(paper['full_text']['section_name'])}")
    print(f"  Questions: {len(qas['question'])}")
    for j, q in enumerate(qas['question'][:3]):
        answers = qas['answers'][j]
        for a in answers['answer']:
            if a['unanswerable']:
                ans_text = "[unanswerable]"
            elif a['extractive_spans']:
                ans_text = " | ".join(a['extractive_spans'])
            elif a['free_form_answer']:
                ans_text = a['free_form_answer']
            elif a['yes_no'] is not None:
                ans_text = "Yes" if a['yes_no'] else "No"
            else:
                ans_text = "[no answer]"
        print(f"    Q{j+1}: {q[:80]}")
        print(f"        Answer: {ans_text[:100]}")


--- Paper 0: Minimally Supervised Learning of Affective Events Using Discourse Relations ---
  Sections: 21
  Questions: 9
    Q1: What is the seed lexicon?
        Answer: seed lexicon consists of positive and negative predicates
    Q2: What are the results?
        Answer: Using all data to train: AL -- BiGRU achieved 0.843 accuracy, AL -- BERT achieved 0.863 accuracy, AL
    Q3: How are relations used to propagate polarity?
        Answer: cause relation: both events in the relation should have the same polarity; concession relation: even

--- Paper 1: PO-EMO: Conceptualization, Annotation, and Modeling of Aesthetic Emotions in Ger ---
  Sections: 25
  Questions: 3
    Q1: Does the paper report macro F1?
        Answer: Yes
    Q2: How is the annotation experiment evaluated?
        Answer: confusion matrices of labels between annotators
    Q3: What are the aesthetic emotions formalized?
        Answer: feelings of suspense experienced in narratives not only respond to the trajec

In [19]:
# Summary statistics across all 10 samples
stats = []
for i in range(len(dataset)):
    paper = dataset[i]
    num_sections = len(paper['full_text']['section_name'])
    num_paragraphs = sum(len(p) for p in paper['full_text']['paragraphs'])
    num_questions = len(paper['qas']['question'])
    total_answers = sum(len(a['answer']) for a in paper['qas']['answers'])
    stats.append({
        "Paper": paper['title'][:50] + "...",
        "Sections": num_sections,
        "Paragraphs": num_paragraphs,
        "Questions": num_questions,
        "Total_Answers": total_answers,
    })

stats_df = pd.DataFrame(stats)
print(stats_df.to_string(index=False))
print(f"\n--- Averages ---")
print(f"  Sections/paper:   {stats_df['Sections'].mean():.1f}")
print(f"  Paragraphs/paper: {stats_df['Paragraphs'].mean():.1f}")
print(f"  Questions/paper:  {stats_df['Questions'].mean():.1f}")

                                                Paper  Sections  Paragraphs  Questions  Total_Answers
Minimally Supervised Learning of Affective Events ...        21          72          9             12
PO-EMO: Conceptualization, Annotation, and Modelin...        25          80          3              4
Community Identity and User Engagement in a Multi-...        14          78          6              8
Question Answering based Clinical Text Structuring...        19          47         12             16
  Progress and Tradeoffs in Neural Language Models...         7          26          3              4
Stay On-Topic: Generating Context-specific Fake Re...         5         644          6              8
Saliency Maps Generation for Automatic Text Summar...        12          34          3              4
  Probabilistic Bias Mitigation in Word Embeddings...        14          52          3              4
Massive vs. Curated Word Embeddings for Low-Resour...        15          58       

In [20]:
# Deep dive into answer types and evidence
from collections import Counter

answer_types = []
evidence_counts = []
for i in range(len(dataset)):
    for ans_list in dataset[i]['qas']['answers']:
        for ans in ans_list['answer']:
            if ans['unanswerable']:
                answer_types.append('unanswerable')
            elif ans['extractive_spans']:
                answer_types.append('extractive')
            elif ans['free_form_answer']:
                answer_types.append('free_form')
            elif ans['yes_no'] is not None:
                answer_types.append('yes_no')
            else:
                answer_types.append('other')
            evidence_counts.append(len(ans['evidence']))

type_counts = Counter(answer_types)
print("Answer Type Distribution:")
for atype, count in type_counts.most_common():
    print(f"  {atype:15s}: {count:3d} ({count/len(answer_types)*100:.1f}%)")
print(f"\nAvg evidence paragraphs per answer: {sum(evidence_counts)/len(evidence_counts):.2f}")

Answer Type Distribution:
  extractive     :  32 (47.1%)
  free_form      :  18 (26.5%)
  unanswerable   :  11 (16.2%)
  yes_no         :   7 (10.3%)

Avg evidence paragraphs per answer: 1.31


## Characters vs. Tokens

In [5]:
from transformers import AutoTokenizer

# We use the specific tokenizer for our embedding model (BGE)
# If we used a GPT tokenizer, the counts would be wrong!
tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-small-en-v1.5")

text = "RAG is a technique that combines retrieval and generation."
chars = len(text)
tokens = len(tokenizer.encode(text))

print(f"Character count: {chars}")
print(f"Token count: {tokens}")
print(f"Ratio: {chars / tokens:.2f} chars per token (approx)")

Character count: 58
Token count: 12
Ratio: 4.83 chars per token (approx)


## The Sliding Window Effect

In [ ]:
# Create a realistic text where each word is unique and distinguishable
long_text = (
    "Retrieval augmented generation combines information retrieval with language models "
    "to produce grounded answers from external knowledge sources. This technique first "
    "retrieves relevant documents from a corpus and then feeds them as context to the "
    "generator model. The result is more factual and verifiable output compared to pure "
    "generation approaches that rely solely on parametric memory stored during training."
)

def simulate_window(text, max_len=20, overlap=5):
    tokens = tokenizer.encode(text, add_special_tokens=False)
    stride = max_len - overlap  # The step size

    print(f"Total Tokens: {len(tokens)}")
    print(f"Window Size: {max_len}, Overlap: {overlap}, Stride: {stride}")

    chunks = []
    for i in range(0, len(tokens), stride):
        chunk_tokens = tokens[i : i + max_len]
        chunks.append(tokenizer.decode(chunk_tokens))
    return chunks

chunks = simulate_window(long_text)
print("-" * 60)
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: \"{chunk}\"")

# Highlight the overlap between consecutive chunks
print("\n" + "=" * 60)
print("OVERLAP VERIFICATION:")
for i in range(len(chunks) - 1):
    words_c1 = chunks[i].split()
    words_c2 = chunks[i + 1].split()
    tail = " ".join(words_c1[-overlap:])
    head = " ".join(words_c2[:overlap])
    print(f"\n  Chunk {i+1} tail:  \"{tail}\"")
    print(f"  Chunk {i+2} head:  \"{head}\"")
    print(f"  Match: {tail == head}")

Total Tokens: 100
Window Size: 20, Overlap: 5, Stride: 15
--------------------
Chunk 1: word word word word word word word word word word word word word word word word word word word word
Chunk 2: word word word word word word word word word word word word word word word word word word word word
